<a href="https://colab.research.google.com/github/m-ony/AhorcadoPokemon/blob/main/Afina_Bert_181619.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Afinación de BERT para tareas específicas

Monica Hernandez Quiroz - 181619

BertForSequenceClassification: Una versión de BERT que incluye una capa final (Linear/Dense layer) para clasificar el texto en N clases.

Trainer y TrainingArguments: Herramientas de Hugging Face para entrenar y evaluar modelos de forma automática sin escribir bucles de entrenamiento manuales en PyTorch.

In [12]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch

# Sample text corpus and labels
documents = [
    "Natural Language Processing analyzes human language.",
    "Text classification is an important NLP task.",
    "Tokenization is used in Natural Language Processing.",
    "Artificial Intelligence can be used in autonomous vehicles.",
    "Machine learning can help predict house prices.",
    "Computer vision allows computers to analyze images.",
    "Robots can use artificial intelligence to make decisions.",
    "Deep learning is widely used in computer vision."
]

labels = [
    1,  # NLP
    1,  # NLP
    1,  # NLP
    0,  # AI
    0,  # AI
    0,  # AI
    0,  # AI
    0   # AI
]  # 1 for NLP-related, 0 for AI-related

# Load pre-trained BERT tokenizer and model for sequence classification
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Tokenize the text data
inputs = tokenizer(documents, padding=True, truncation=True, return_tensors='pt')

# Create a dataset class
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.inputs.items()}
        item['labels'] = self.labels[idx]
        return item

# Split the data into training and testing sets
#El error esta aqui en train_test_split, porque está diseñada para operar sobre listas de Python o arreglos NumPy. No con imputs que es un diccionario especial de PyTorch.
#train_inputs, test_inputs, train_labels, test_labels = train_test_split(inputs, labels, test_size=0.2, random_state=42)
train_text, test_text, train_labels, test_labels = train_test_split(documents, labels, test_size=0.2, random_state=42)

#Vamos a tokenizar
train_inputs = tokenizer(train_text, padding=True, truncation=True, return_tensors='pt')
test_inputs = tokenizer(test_text, padding=True, truncation=True, return_tensors='pt')

# Create datasets
train_dataset = TextDataset(train_inputs, train_labels)
test_dataset = TextDataset(test_inputs, test_labels)

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results', # carpeta donde se guardan los putos de control del modelo
    num_train_epochs=3, # El número de épocas
    per_device_train_batch_size=4, # El número de muestras (oraciones) procesadas al mismo tiempo
    per_device_eval_batch_size=4, # El número de muestras procesadas en paralelo durante la evaluación
    warmup_steps=10, # Número de pasos iniciales en los que la tasa de aprendizaje
    weight_decay=0.01, # Aplica una penalización L2 a los pesos grandes del modelo.
    #Eliminamos esta linea por que ya esta desactualizada
    #logging_dir='./logs', #La carpeta donde se almacenan los registros (logs) del entrenamiento (métricas de pérdida, tasa de aprendizaje, tiempo por época, etc.).
    logging_steps=10, #Indica cada cuántos pasos se imprimen en consola.

)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print("Evaluation results:")
print(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
No log,0.664638,6


Evaluation results:
{'eval_loss': 0.6646376252174377}


El resultado final de la evaluación. Indica que la función de pérdida del modelo en el conjunto de prueba es de aproximadamente 0.75